# TopoPolynormer: giving a graph transformer topological sight

**The idea in one sentence.** Message passing is a whispering game along edges, so it can never tell whether *two of your neighbours are also neighbours of each other* — i.e. whether you sit on a **triangle**. We hand each node that missing fact.

This is not a hack; it is the canonical expressivity gap. Standard message-passing GNNs are bounded by the 1-Weisfeiler-Leman test, which **provably cannot count triangles** ([Chen et al., NeurIPS 2020](https://arxiv.org/abs/2002.04025)). Adding substructure counts as input features provably lifts a model beyond 1-WL ([Graph Substructure Networks, Bouritsas et al.](https://arxiv.org/abs/2006.09252)). A triangle is a **2-simplex**, so this is the minimal, literal bridge from a graph (edges = 1-simplices) to topology (triangles = 2-simplices) — exactly the theme of the challenge.

**TopoPolynormer** = the Polynormer graph transformer ([Deng, Yue & Zhang, ICLR 2024](https://arxiv.org/abs/2403.01232)) + a small per-node **structural fingerprint** computed from `edge_index`: `[log1p(degree), triangle count, log1p(triangle), clustering coefficient, random-walk return probabilities at steps 2,3,4]`. The triangle/clustering channels supply the topological signal; the degree and multi-scale random-walk channels keep the encoding informative and scale-stable across the GraphUniverse degree/density regimes (where triangles can become rare).

## 1. The structural encoding, and why it is exactly correct

We verify two things against `networkx`: (a) the triangle channel equals each node's true triangle count, and (b) — the property that makes graph-level triangle counting almost trivial — the **sum** of per-node triangle counts is exactly `3 x (total triangles)`. Since the challenge readout is sum-pooling over nodes, the model gets a near-linear path to the triangle-counting target.

In [ ]:
import networkx as nx
import torch
from torch_geometric.data import Batch, Data

from topobench.nn.backbones.graph.topo_polynormer import (
    TRIANGLE_SCALE,
    TopoPolynormer,
    structural_encoding,
)

torch.manual_seed(0)

ei = torch.randint(0, 24, (2, 90))
ei = ei[:, ei[0] != ei[1]]
ei = torch.cat([ei, ei.flip(0)], dim=1)  # undirected

s = structural_encoding(ei, num_nodes=24, batch=None)
tri_model = s[:, 1] * TRIANGLE_SCALE          # undo the fixed scaling

G = nx.Graph(); G.add_nodes_from(range(24)); G.add_edges_from(ei.t().tolist())
tri_nx = torch.tensor([nx.triangles(G)[i] for i in range(24)], dtype=torch.float)

print('per-node triangle channel == networkx:', torch.allclose(tri_model, tri_nx, atol=1e-4))
print('sum of per-node triangles:', tri_model.sum().item(), '= 3 x', sum(nx.triangles(G).values()) // 3)
print('encoding channels per node:', s.shape[1])

## 2. The encoding is batch-aware

Substructure counts are computed per graph segment, so batching disjoint graphs never mixes their structure. We check that a graph's fingerprint is identical alone vs. inside a batch.

In [ ]:
g_a = Data(x=torch.randn(24, 16), edge_index=ei, num_nodes=24)
g_b = Data(x=torch.randn(13, 16), edge_index=torch.randint(0, 13, (2, 40)), num_nodes=13)
batch = Batch.from_data_list([g_a, g_b])
s_batched = structural_encoding(batch.edge_index, batch.num_nodes, batch.batch)
print('structural encoding batch-isolation (A alone == A in batch):',
      torch.allclose(s, s_batched[:24], atol=1e-5))

## 3. The backbone

TopoPolynormer maps node features to embeddings; the TopoBench readout produces the final logits. The structural encoding is injected *inside* the backbone (after the input projection), which bypasses the feature encoder's GraphNorm so the absolute triangle-count scale is preserved.

In [ ]:
model = TopoPolynormer(
    in_channels=16, hidden_channels=32, out_channels=16,
    local_layers=3, global_layers=2, heads=1, use_struct=True,
).eval()
out = model(g_a.x, g_a.edge_index, batch=torch.zeros(24, dtype=torch.long))
print('output shape:', tuple(out.shape))
print('parameters  :', sum(p.numel() for p in model.parameters()))

## 4. Training in the TopoBench pipeline

The model ships with `configs/model/graph/topo_polynormer.yaml`, so the canonical run is `python -m topobench model=graph/topo_polynormer dataset=<dataset>`. Here we train briefly on MUTAG from the notebook.

In [ ]:
import os
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from topobench.run import run
from topobench.utils.config_resolvers import register_all_resolvers

root = Path.cwd().resolve()
while not (root / 'configs' / 'run.yaml').exists():
    root = root.parent
os.environ['PROJECT_ROOT'] = str(root)
out_dir = root / 'logs' / 'tutorial_topo_polynormer'
out_dir.mkdir(parents=True, exist_ok=True)

register_all_resolvers()
GlobalHydra.instance().clear()
with initialize_config_dir(version_base='1.3', config_dir=str(root / 'configs')):
    cfg = compose(
        config_name='run.yaml',
        overrides=[
            'model=graph/topo_polynormer',
            'dataset=graph/MUTAG',
            'logger=csv',
            'trainer.max_epochs=10',
            'trainer.accelerator=cpu',
            'trainer.devices=1',
            '+trainer.enable_progress_bar=False',
            f'paths.output_dir={out_dir.as_posix()}',
            f'paths.work_dir={root.as_posix()}',
        ],
    )

metric_dict, _ = run(cfg)
print('\nMetrics:')
for key, value in metric_dict.items():
    print(f'{key:<20s} {float(value):.4f}')

We restored the one structural signal a message-passing GNN provably cannot compute — triangle (2-simplex) participation — verified it is exact, confirmed it is batch-aware, and trained the model end-to-end. To benchmark on the challenge's GraphUniverse datasets, set `MODEL_CONFIG = "graph/topo_polynormer"` in `2026_tdl_challenge/run_evaluation.ipynb`.